[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/curvature/curvature.ipynb)

# Curvature of a Gravitational Wave

This example computes the spacetime curvature of a passing gravitational wave and the tidal motion it raises across a ring of freely falling beads in plus, cross, and circular polarizations. Curvature governs how neighbouring paths of free fall bend toward or away from one another, and a local observer measures it as a relative acceleration proportional to spatial separation. A vacuum plane wave is modelled as a linear map from spacetime area elements to the bivector planes they turn in.

Joining the wave vector with two transverse slices builds a pair of null planes whose opposite-signed dyads form the metric-free curvature map `(wave ^ x) * ((wave ^ x) & Antibivector) - (wave ^ y) * ((wave ^ y) & Antibivector)`. Because every output plane contains the wave vector, `wave ^ plus` vanishes identically, and rotating input and output across the wave produces the cross polarization `eighth_turn >> plus(eighth_turn << Antibivector)` while reversing sign after a quarter turn. Pairing an observer's four-velocity twice extracts the tidal map `curvature(observer & Antivector) & observer`, and the same wave acts directly on separations as a strain map whose second derivative along `wave` reconstructs the curvature.

In the tensor notation of general relativity, spacetime curvature is represented by the four-index Riemann tensor $R_{\alpha\beta\gamma\delta}$ subject to $R_{\alpha\beta\gamma\delta} = -R_{\beta\alpha\gamma\delta} = -R_{\alpha\beta\delta\gamma}$, $R_{\alpha\beta\gamma\delta} = R_{\gamma\delta\alpha\beta}$, $R_{\alpha[\beta\gamma\delta]} = 0$, and the vacuum trace condition $R_{\beta\delta} = g^{\alpha\gamma} R_{\alpha\beta\gamma\delta} = 0$, with Petrov type N written $k_{[\alpha} R_{\beta\gamma]\delta\epsilon} = 0$. Tidal acceleration follows the geodesic deviation equation $D^2 \xi^\alpha / d\tau^2 = -R^\alpha{}_{\beta\gamma\delta} u^\beta \xi^\gamma u^\delta$, where linearized plane waves in transverse-traceless gauge relate curvature to the metric perturbation $h^{\mathrm{TT}}_{ij}$ through $R_{\alpha\beta\gamma\delta} = \tfrac{1}{2}(\partial_\beta\partial_\gamma h_{\alpha\delta} + \partial_\alpha\partial_\delta h_{\beta\gamma} - \partial_\alpha\partial_\gamma h_{\beta\delta} - \partial_\beta\partial_\delta h_{\alpha\gamma})$ and $R_{0i0j} = -\tfrac{1}{2}\ddot{h}^{\mathrm{TT}}_{ij}$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import STA
from examples.relativity.curvature import render
from examples.util.animation import save_figure, show_animation

mv = NumpyContext(STA).multivector
Scalar = STA.gatype.scalar()                        # Scalar+
# Wave vectors and the planes fields and curvature turn in are extrinsic vectors and bivectors.
Vector = STA.gatype.vector()                        # Vector+
Bivector = STA.gatype.bivector()                    # Bivector+
Trivector = STA.gatype.trivector()                  # Trivector+
# Events, velocities, separations, and the areas their joins span are intrinsic antivectors and antibivectors.
Antivector = STA.gatype.antivector()                # Antivector-
Antibivector = STA.gatype.antibivector()            # Antibivector-
Rotor = STA.gatype.rotor()                          # Rotor+

Curvature = STA.gatype((Bivector, Antibivector))    # Bivector+ <- Antibivector-
Tidal = STA.gatype((Vector, Antivector))            # Vector+ <- Antivector-
Strain = STA.gatype((Vector, Antivector))           # Vector+ <- Antivector-
# Curvature read on an ordered pair of plane edges.
CurvatureForm = STA.gatype((Bivector, Antivector, Antivector))   # Bivector+ <- (Antivector-, Antivector-)

# One unit along laboratory time, perpendicular to the spatial slice t.
time_direction = -mv.t.dual()                       # [] Antivector-
# Null propagation along +z and its dual wave vector.
travel = time_direction + mv.z.dual()               # [] Antivector-
wave_vector = -travel.dual()                        # [] Vector+
# Eighth-turn rotor in the transverse xy plane.
eighth_turn = (mv.xy * (-np.pi / 8)).exp()          # [] Rotor+

## Curvature from null dyads

A plane wave propagating at the speed of light carries a null wave vector that wedges with two orthogonal transverse slices to form two null bivector planes. Weighting each null plane by its regressive pairing with an area builds a metric-free curvature map from areas to planes, and giving the two dyads opposite signs cancels the vacuum trace. Every output plane contains the wave vector so `wave_vector ^ plus` vanishes across the entire space of areas, while conjugating input and output by `eighth_turn` yields the cross polarization and squaring that rotation reverses `plus`, exhibiting spin-two symmetry.

In [ ]:
def plane_wave_curvature(wave: Vector, first_slice: Vector, second_slice: Vector) -> Curvature:
    """Vacuum plane-wave curvature maps spacetime areas onto the null planes of `wave` with `first_slice` and `second_slice`."""
    first_plane = wave ^ first_slice                # [] Bivector+
    second_plane = wave ^ second_slice              # [] Bivector+
    return first_plane * (first_plane & Antibivector) - second_plane * (second_plane & Antibivector)   # [] Bivector+ <- Antibivector-


def polarization_curvatures() -> tuple[Curvature, Curvature]:
    """Unit plus and cross curvature maps propagate along +z and differ by an eighth-turn in the transverse plane."""
    plus = plane_wave_curvature(wave_vector, mv.x, mv.y)                # [] Bivector+ <- Antibivector-
    cross = eighth_turn >> plus(eighth_turn << Antibivector)            # [] Bivector+ <- Antibivector-
    return plus, cross


plus, cross = polarization_curvatures()                                 # [] Bivector+ <- Antibivector-, Bivector+ <- Antibivector-
quarter_turn = eighth_turn * eighth_turn                                # [] Rotor+
type_n = wave_vector ^ plus                                             # [] Trivector+ <- Antibivector-
spin_two = (quarter_turn >> plus(quarter_turn << Antibivector)) + plus  # [] Bivector+ <- Antibivector-

## Symmetries of the curvature

Pair symmetry and the first Bianchi identity hold directly on the metric-free curvature map, while the vacuum condition enters once an observer and the metric convert curvature into a tidal quadratic form. Pairing the output plane of one area with a second area defines a symmetric bilinear form on areas, and cycling three directions through a fixed pair sum yields the zero map from a third direction to covectors. Boosted observers moving along any spatial axis measure traceless tidal forms whose stretching and squeezing eigenvalues sum to zero.

In the tensor notation of general relativity, these three identities read as $R_{\alpha\beta\gamma\delta} = R_{\gamma\delta\alpha\beta}$, $R_{\alpha[\beta\gamma\delta]} = 0$, and $R_{\beta\delta} = R^\alpha{}_{\beta\alpha\delta} = 0$.

In [ ]:
edge_a, edge_b, edge_c, edge_d = mv.vector(np.random.default_rng(7).normal(size=(4, 4))).dual()   # [] Antivector-, Antivector-, Antivector-, Antivector-
first_area = edge_a & edge_b                                            # [] Antibivector-
second_area = edge_c & edge_d                                           # [] Antibivector-

# Pair symmetry equates the reciprocal pairings of two areas across the curvature map.
pair_symmetry = (plus(second_area) & first_area) - (plus(first_area) & second_area)   # [] Scalar+

# Cycling the three edges of a volume sends the summed map on the third direction to zero.
bianchi = (plus(first_area) & Antivector) + (plus(edge_b & Antivector) & edge_a) + (plus(Antivector & edge_a) & edge_b)   # [] Vector+ <- Antivector-

# Boosted observers along all three spatial axes measure traceless tidal quadratic forms.
spatial_axes = stack((mv.x.dual(), mv.y.dual(), mv.z.dual()))           # [3] Antivector-
axis_boosts = ((spatial_axes & time_direction).dual() * 0.3).exp()      # [3] Rotor+
symmetry_observers = axis_boosts >> time_direction                      # [3] Antivector-
tides = (plus(symmetry_observers & Antivector) & symmetry_observers)(Antivector) & Antivector   # [3] Scalar+ <- (Antivector-, Antivector-)
vacuum_traces = tides.eigvals(Antivector.dot(Antivector)).sum(axis=-1)  # [3] Scalar+

## Tidal map of an observer

Joining an observer's four-velocity with a spatial separation spans a narrow spacetime ribbon `observer & Antivector`, and pairing the resulting curvature plane a second time with the observer extracts the tidal force covector `curvature(observer & Antivector) & observer`. Spacetime duality converts that force covector into a relative acceleration within the observer's rest slice. Measured against the metric `Antivector.dot(Antivector)`, the tidal quadratic form of a unit plus wave has opposite unit eigenvalues along the two transverse directions and zero eigenvalues along time and the propagation direction.

In the tensor notation of general relativity, relative free fall obeys the geodesic deviation equation $\frac{D^2 \xi^\alpha}{d\tau^2} = -R^\alpha{}_{\beta\gamma\delta}\, u^\beta \xi^\gamma u^\delta$ for four-velocity $u^\beta$ and separation $\xi^\gamma$.

In [ ]:
def tidal_map(curvature: Curvature, observer: Antivector) -> Tidal:
    """Double pairing of `curvature` with `observer` maps spatial separations to tidal force covectors."""
    return curvature(observer & Antivector) & observer                  # [...] Vector+ <- Antivector-


tidal_plus = tidal_map(plus, time_direction)                            # [] Vector+ <- Antivector-
tidal_eigenvalues = (tidal_plus(Antivector) & Antivector).eigvals(Antivector.dot(Antivector))   # [4] Scalar+

save_figure(render.curvature_map_figure(plus, time_direction, travel, mv.x.dual()), "curvature_map");

The left diagram shows the translucent spacetime ribbon spanned by the laboratory observer's orange velocity arrow along `ct` and a unit separation along `x`, while the right diagram shows its image under `plus` as the null plane spanned by the blue propagation arrow and the orange tidal acceleration arrow.

## Doppler shift for boosted observers

Observers boosted along the propagation axis experience a relativistic Doppler shift in wave frequency, redshifted when chasing the wave and blueshifted when flying into it. Because `tidal_map` pairs the observer's velocity with the curvature twice, the measured tidal amplitude scales with the square of the Doppler factor, decaying exponentially with twice the rapidity. Applying boost rotors to the laboratory rest velocity builds a batch of observer four-velocities whose acceleration maps carry that scaling in their leading singular values.

In special-relativistic rapidity notation, a wave of rest amplitude $A$ boosted by rapidity $\zeta$ along the propagation axis has tidal amplitude $A\, e^{-2\zeta}$.

In [ ]:
max_rapidity = 0.7
n_observers = 15

In [ ]:
def boosted_observers(rapidities: np.ndarray, direction: Antivector) -> Antivector:
    """Lorentz boosts along `direction` carry the laboratory rest velocity to one four-velocity per rapidity."""
    # The join of the spatial direction with time spans the boost ribbon, whose dual generates the boost.
    boosts = ((direction & time_direction).dual() * (rapidities / 2)).exp()   # [...] Rotor+
    return boosts >> time_direction                                     # [...] Antivector-


rapidities = np.linspace(-max_rapidity, max_rapidity, n_observers)
chasing_observers = boosted_observers(rapidities, mv.z.dual())          # [n_observers] Antivector-
responses = tidal_map(plus, chasing_observers)                          # [n_observers] Vector+ <- Antivector-
amplitudes = (-responses.dual()).svdvals()[..., 0]                      # [n_observers] Scalar+

save_figure(render.doppler_figure(rapidities, amplitudes), "curvature_doppler");

Blue markers show the leading singular values of the boosted acceleration maps across negative and positive rapidities, falling directly along the dark exponential Doppler curve on both sides of the faint rest line at zero rapidity.

## Wave packet in three polarizations

A localized wave packet modulates a sinusoidal carrier by a smooth Gaussian envelope, with the cross carrier lagging a quarter cycle behind the plus carrier. Combining both quadrature profiles with equal weight forms a circularly polarized wave whose principal axes rotate steadily at half the carrier rate. Weighting the unit `plus` and `cross` curvature maps by minus half the profiles' second time derivatives assembles a single curvature batch across time and polarization.

In linearized general relativity, the electric components of the Riemann tensor in transverse-traceless gauge satisfy $R_{0i0j} = -\tfrac{1}{2}\ddot{h}^{\mathrm{TT}}_{ij}$.

In [ ]:
wave_duration = 6.0
n_cycles = 9
wave_amplitude = 1e-4
n_samples = 1201
polarization_weights = np.array([[1, 0], [0, 1], [1, 1]])

In [ ]:
def wave_packet(
    time: np.ndarray, duration: float, n_cycles: int, amplitude: float,
) -> tuple[np.ndarray, np.ndarray]:
    """A complex Gaussian carrier evaluates the plus and cross strain profiles (`[..., 2]`) and their second derivatives."""
    centered_time = time - duration / 2
    envelope_width = duration / 12
    carrier_rate = 2 * np.pi * n_cycles / duration
    # Both quadrature profiles and their second derivatives follow from one complex Gaussian carrier.
    complex_profile = amplitude * np.exp(-centered_time**2 / (2 * envelope_width**2) + 1j * carrier_rate * centered_time)
    complex_second = ((-centered_time / envelope_width**2 + 1j * carrier_rate) ** 2 - 1 / envelope_width**2) * complex_profile
    strains = np.stack([complex_profile.real, complex_profile.imag], axis=-1)
    second_derivatives = np.stack([complex_second.real, complex_second.imag], axis=-1)
    return strains, second_derivatives


time = np.linspace(0.0, wave_duration, n_samples)
profiles, second_derivatives = wave_packet(time, wave_duration, n_cycles, wave_amplitude)
curvature_weights = -0.5 * second_derivatives[:, None] * polarization_weights
waves = curvature_weights[..., 0] * plus + curvature_weights[..., 1] * cross   # [n_samples, 3] Bivector+ <- Antibivector-

save_figure(render.packet_figure(time, profiles, -0.5 * second_derivatives), "curvature_packet");

The upper strip plots the quadrature plus and cross strain envelopes in blue and orange, and the lower strip plots the corresponding curvature weights proportional to minus half the second time derivative.

## Ring of freely falling beads

Binding the laboratory rest observer to the batched curvature produces a tidal map at every sample time and polarization. When the detector ring is much smaller than the carrier wavelength, each bead experiences the tidal force evaluated at its rest separation on the ring. Converting the force covectors to spatial accelerations via `.dual()` and integrating twice from rest with symmetric cubic quadrature traces the beads' trajectories across all three polarizations.

In [ ]:
ring_radius = 0.01
n_beads = 24
amplification = 4000

In [ ]:
def cumulative_integral(step: float, values: Antivector) -> Antivector:
    """Symmetric cubic quadrature accumulates the time integral of `values` along the first batch axis from zero."""
    zero = 0 * values[:1]                                               # [...] Antivector-
    if values.shape[0] < 4:
        steps = (step / 2) * (values[:-1] + values[1:])                 # [...] Antivector-
        return concatenate((zero, steps)).cumsum(axis=0)                # [...] Antivector-
    first_step = (step / 24) * (9 * values[0:1] + 19 * values[1:2] - 5 * values[2:3] + values[3:4])   # [...] Antivector-
    interior_steps = (step / 24) * (-values[:-3] + 13 * values[1:-2] + 13 * values[2:-1] - values[3:])   # [...] Antivector-
    last_step = (step / 24) * (values[-4:-3] - 5 * values[-3:-2] + 19 * values[-2:-1] + 9 * values[-1:])   # [...] Antivector-
    return concatenate((zero, first_step, interior_steps, last_step)).cumsum(axis=0)   # [...] Antivector-


def integrated_displacements(time: np.ndarray, acceleration: Antivector) -> Antivector:
    """Repeated cumulative quadrature integrates `acceleration` twice from rest along the uniform `time` axis."""
    if len(time) < 2:
        return 0 * acceleration                                         # [...] Antivector-
    step = float(time[1] - time[0])
    velocities = cumulative_integral(step, acceleration)                # [...] Antivector-
    return cumulative_integral(step, velocities)                        # [...] Antivector-


response = tidal_map(waves, time_direction)                             # [n_samples, 3] Vector+ <- Antivector-
angles = np.linspace(0, 2 * np.pi, n_beads, endpoint=False)
bead_turns = (mv.xy * (-angles / 2)).exp()                              # [n_beads] Rotor+
reference = bead_turns >> (mv.x.dual() * ring_radius)                   # [n_beads] Antivector-
acceleration = -response[:, :, None](reference).dual()                  # [n_samples, 3, n_beads] Antivector-
displacement = integrated_displacements(time, acceleration)             # [n_samples, 3, n_beads] Antivector-

save_figure(render.detector_figure(time, reference, displacement, acceleration, amplification), "curvature_detector");

The three panels show the plus, cross, and circular bead rings at peak deformation against the dotted rest circle, with blue tidal acceleration arrows on eight beads and faint full-packet trajectories on four highlighted beads.

## Strain map and linearized curvature

The same wave acts directly on rest separations as a transverse strain map of type `Vector <- Antivector`, built from `x * (x & Antivector) - y * (y & Antivector)` and its eighth-turn conjugate. Applying spacetime duality to `strain_map(reference)` yields each bead's displacement directly without time integration, while comparing `response` against minus the strain's second time derivative relates two maps of identical type without invoking the metric. Wedging that second derivative with `wave_vector` and antisymmetrizing over two ordered edge arguments reconstructs the full plane-wave curvature from the strain.

In linearized general relativity, the transverse-traceless strain map corresponds to half the metric perturbation $\tfrac{1}{2} h^{\mathrm{TT}}_{ij}$, and the curvature tensor follows from $R_{\alpha\beta\gamma\delta} = \tfrac{1}{2}(\partial_\beta\partial_\gamma h_{\alpha\delta} + \partial_\alpha\partial_\delta h_{\beta\gamma} - \partial_\alpha\partial_\gamma h_{\beta\delta} - \partial_\beta\partial_\delta h_{\alpha\gamma})$.

In [ ]:
def strain_patterns() -> tuple[Strain, Strain]:
    """Unit plus and cross transverse strain maps send spatial separations to the covectors reading their stretch."""
    plus_strain = mv.x * (mv.x & Antivector) - mv.y * (mv.y & Antivector)   # [] Vector+ <- Antivector-
    cross_strain = eighth_turn >> plus_strain(eighth_turn << Antivector)    # [] Vector+ <- Antivector-
    return plus_strain, cross_strain


def curvature_of_strain(wave: Vector, second_strain: Strain) -> CurvatureForm:
    """Wedging `second_strain` with `wave` builds the linearized curvature map on ordered pairs of directions."""
    bent_slice = second_strain ^ wave                                   # [...] Bivector+ <- Antivector-
    return (wave & Antivector) * bent_slice - bent_slice * (wave & Antivector)   # [...] Bivector+ <- (Antivector-, Antivector-)


plus_strain, cross_strain = strain_patterns()                           # [] Vector+ <- Antivector-, Vector+ <- Antivector-
strain_weights = 0.5 * profiles[:, None] * polarization_weights
strain_map = strain_weights[..., 0] * plus_strain + strain_weights[..., 1] * cross_strain   # [n_samples, 3] Vector+ <- Antivector-

# Spacetime duality converts each bead's stretch reading directly into its spatial displacement.
predicted = strain_map[:, :, None](reference).dual()                    # [n_samples, 3, n_beads] Antivector-
displacement_error = (stack((mv.t, mv.x, mv.y, mv.z)) & (displacement - predicted)[..., None]).abs()   # [n_samples, 3, n_beads, 4] Scalar+

second_weights = 0.5 * second_derivatives[:, None] * polarization_weights
second_strain = second_weights[..., 0] * plus_strain + second_weights[..., 1] * cross_strain   # [n_samples, 3] Vector+ <- Antivector-
tidal_from_strain = response + second_strain                            # [n_samples, 3] Vector+ <- Antivector-
curvature_two_form = curvature_of_strain(wave_vector, second_strain)    # [n_samples, 3] Bivector+ <- (Antivector-, Antivector-)

sample_edge = mv.vector(np.random.default_rng(5).normal(size=4)).dual()   # [] Antivector-
strain_curvature_residual = (curvature_two_form(sample_edge) - waves(sample_edge & Antivector)).svdvals()   # [n_samples, 3, 4] Scalar+

## Circular wave through the bead ring

Released from rest ahead of the Gaussian envelope, the beads return to rest once the wave packet passes. Under circular polarization, the elliptical ring profile rotates continuously while each individual bead traces a small local orbit around its own rest position on the circle.

In [ ]:
frame_duration_ms = 40
shown = slice(n_samples // 4, 3 * n_samples // 4 + 1)

In [ ]:
show_animation(
    render.detector_animation(time[shown], reference, displacement[shown], acceleration[shown], amplification),
    "curvature",
    frame_duration_ms,
)

As the circular packet sweeps past, the blue ellipse and its inward-outward tidal acceleration field rotate steadily while the four colour-coded beads trace local circular loops around their rest positions on the dotted ring.

In [ ]:
# checks
assert (type_n.svdvals() < 1e-10).all()
assert (spin_two.svdvals() < 1e-10).all()
assert (pair_symmetry.abs() < 1e-10).all()
assert (bianchi.svdvals() < 1e-10).all()
assert (vacuum_traces.abs() < 1e-10).all()
assert ((tidal_eigenvalues * (tidal_eigenvalues - 1) * (tidal_eigenvalues + 1)).abs() < 1e-10).all()
assert (tidal_eigenvalues.sum().abs() < 1e-10).all()
assert ((tidal_eigenvalues.abs().sum() - 2).abs() < 1e-10).all()
assert ((amplitudes - np.exp(-2 * rapidities)).abs() < 1e-10).all()
assert (displacement_error < 1e-7).all()
assert (tidal_from_strain.svdvals() < 1e-10).all()
assert (strain_curvature_residual < 1e-10).all()